# Notebook 0: Setup and Environment Verification

Welcome to the BINFX410 Spark Genomics Exercise!

This notebook verifies that your environment is set up correctly. Work through each cell in order.

**Estimated time: 10-15 minutes**

## Step 1: Import Libraries

Run the cell below to verify all required Python packages are installed.

In [ ]:
# Test all required imports
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

print(f"PySpark version: {pyspark.__version__}")
print(f"Pandas version:  {pd.__version__}")
print(f"NumPy version:   {np.__version__}")
print("\nAll imports successful!")

## Step 2: Create a SparkSession

A **SparkSession** is the entry point to all Spark functionality. Think of it as your
connection to the Spark engine. Key configuration:

- `appName`: A name for your application (appears in the Spark UI)
- `master("local[*]")`: Run locally using all available CPU cores
- `spark.sql.shuffle.partitions`: Number of partitions after shuffles (8 is good for local)
- `spark.driver.memory`: How much RAM Spark can use

**Important**: Only one SparkSession can be active per notebook. If you need to restart,
use `spark.stop()` first, then create a new one.

In [ ]:
spark = SparkSession.builder \
    .appName("BINFX410_Setup_Test") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark UI available at: http://localhost:4040")
print("SparkSession created successfully!")

## Step 3: Hello World - Your First Spark DataFrame

A Spark **DataFrame** is like a pandas DataFrame, but distributed across multiple
partitions. The API is similar but not identical. Let's create a simple one from
a Python list.

In [ ]:
# Create a simple DataFrame from a Python list
data = [
    ("chr1", 12345, "A", "G", 99.0),
    ("chr1", 67890, "C", "T", 45.0),
    ("chr2", 11111, "G", "A", 78.0),
    ("chr2", 22222, "T", "C", 12.0),
]
columns = ["CHROM", "POS", "REF", "ALT", "QUAL"]

df = spark.createDataFrame(data, columns)
df.show()
df.printSchema()
print(f"Row count: {df.count()}")

## Step 4: Basic Operations

Spark DataFrames support operations similar to pandas: filtering, selecting, and grouping.

In [ ]:
# Filter: keep only high-quality variants
high_qual = df.filter(F.col("QUAL") >= 30)
print(f"High quality variants: {high_qual.count()}")
high_qual.show()

# Select specific columns
print("Selected columns:")
df.select("CHROM", "POS").show()

# GroupBy: count variants per chromosome
print("Variants per chromosome:")
df.groupBy("CHROM").count().show()

## Step 5: Generate Test Data

The exercise uses synthetic genomics data generated by a script. Let's generate
a small test dataset to verify everything works.

In [ ]:
import subprocess
import os

data_dir = "/workspace/data"
if not os.path.exists(data_dir):
    data_dir = os.path.join(os.path.dirname(os.getcwd()), "data")

result = subprocess.run(
    ["python", os.path.join(data_dir, "generate_data.py"),
     "--size", "small", "--output-dir", data_dir],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")
else:
    print("Data generation complete!")

## Step 6: Load and Verify Generated Data

In [ ]:
# Load the small variant dataset
variants = spark.read.csv(
    os.path.join(data_dir, "variants_small.csv"),
    header=True,
    inferSchema=True
)

print(f"Loaded {variants.count()} variants")
print(f"\nSchema:")
variants.printSchema()
print(f"\nFirst 5 rows:")
variants.show(5, truncate=False)

In [ ]:
# Summary statistics
print("Summary statistics for numeric columns:")
variants.select("QUAL", "DP", "AF", "GQ").summary().show()

print("\nVariants per chromosome:")
variants.groupBy("CHROM").count().orderBy("count", ascending=False).show()

print("\nFilter distribution:")
variants.groupBy("FILTER").count().orderBy("count", ascending=False).show()

## Step 7: Verify Spark UI

Open **http://localhost:4040** in your web browser. You should see the Spark UI with:

- **Jobs tab**: Lists all completed jobs (you should see several from the operations above)
- **Stages tab**: Shows individual stages within each job
- **Storage tab**: Shows any cached DataFrames (empty for now)
- **SQL tab**: Shows query plans for DataFrame operations

Take a moment to explore the UI - you'll use it throughout the exercises to
understand how Spark executes your queries.

## Troubleshooting

| Problem | Solution |
|---------|----------|
| `ModuleNotFoundError` | Rebuild Docker container: `docker-compose build` |
| Spark UI not loading | Check port 4040 is exposed; may need to wait a few seconds |
| Cannot find data files | Verify Docker volume mounts; check paths |
| Java errors | Ensure Java is installed (should be included in Docker image) |
| `Py4JError` | Restart the kernel and run cells again |

## Success! You're Ready

If all cells above ran without errors, your environment is set up correctly.

**Next step**: Open `01_spark_basics.ipynb` to learn core Spark concepts.

Remember to stop the SparkSession before moving to the next notebook.

In [ ]:
spark.stop()
print("SparkSession stopped. You're ready for the next notebook!")